# Day 7 — Structured Logging & Production Debugging

## Objective
Demonstrate production-grade structured logging setup. Compare `print()` vs `logging`, configure centralized logging with JSON formatting (`JSONFormatter`), record step metrics/durations, and inspect exception tracebacks with `logger.exception()`.

## 1. Logging Setup & Configuration

In [1]:
import sys
import json
from pathlib import Path
repo_root = Path.cwd().resolve()
sys.path.insert(0, str(repo_root / 'src'))

from task_analytics import configure_logging, Pipeline, CleanDataStep, NormalizeDataStep, PriorityFilterStep, PipelineError

log_file = repo_root / 'logs' / 'day7_notebook_demo.log'
logger = configure_logging(level='DEBUG', log_file=log_file, env='production')
logger.info('Structured logging initialized for notebook demo.')

{"timestamp": "2026-09-22T09:52:34.306714+00:00", "level": "INFO", "logger": "task_analytics", "message": "Structured logging initialized for notebook demo.", "module": "<string>", "function": "<module>", "line": 11}


## 2. Successful Pipeline Run (RUN 1)

Execute valid task pipeline and capture structured step start/end metrics.

In [2]:
valid_tasks = [
    {'id': 1, 'title': ' Auth Bug ', 'status': 'in_progress', 'priority': 'HIGH'},
    {'id': 2, 'title': ' DB Migration ', 'status': 'completed', 'priority': 'MEDIUM'},
]
pipeline = Pipeline([
    CleanDataStep(required_keys=['title']),
    NormalizeDataStep(target_fields=['status', 'priority']),
    PriorityFilterStep(priority='high')
])

logger.info('Starting RUN 1 (Success)', extra={'run_id': 'run-001'})
results = pipeline.run(valid_tasks)
logger.info(f'RUN 1 finished with {len(results)} records', extra={'run_id': 'run-001'})
print('Pipeline execution finished successfully.')

Pipeline execution finished successfully.


## 3. Deliberately Failed Pipeline Run (RUN 2)

Trigger a `ProcessingError` and log full traceback using `logger.exception()`.

In [3]:
bad_tasks = [{'id': 101, 'title': 'Task X', 'complexity': 'INVALID_NUMERIC'}]
fail_pipeline = Pipeline([CleanDataStep(required_keys=['title']), NormalizeDataStep(numeric_fields=['complexity'])])

logger.info('Starting RUN 2 (Failure test)', extra={'run_id': 'run-002'})
try:
    fail_pipeline.run(bad_tasks)
except PipelineError as exc:
    logger.exception('Captured expected error in RUN 2: %s', exc, extra={'run_id': 'run-002'})
    print('Caught expected PipelineError and logged traceback.')

Caught expected PipelineError and logged traceback.


## 4. Log File Inspection (`logs/day7_notebook_demo.log`)

Read and parse single-line JSON log records generated by `JSONFormatter`.

In [4]:
if log_file.exists():
    lines = [line.strip() for line in log_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    print(f'Total log entries written: {len(lines)}')
    print('\n--- Sample Log Record (Success Run) ---')
    for line in lines:
        if 'run-001' in line:
            print(json.dumps(json.loads(line), indent=2))
            break

    print('\n--- Sample Log Record (Failure Run with Traceback) ---')
    for line in lines:
        if 'run-002' in line and 'traceback' in line:
            parsed = json.loads(line)
            print('Level:    ', parsed.get('level'))
            print('Error:    ', parsed.get('error_type'), ':', parsed.get('error_message'))
            print('Traceback snippet:')
            for tb_line in parsed.get('traceback', '').splitlines()[-4:]:
                print('  ', tb_line)
            break

Total log entries written: 20

--- Sample Log Record (Success Run) ---
{
  "timestamp": "2026-09-22T09:52:34.309481+00:00",
  "level": "INFO",
  "logger": "task_analytics",
  "message": "Starting RUN 1 (Success)",
  "module": "<string>",
  "function": "<module>",
  "line": 11,
  "run_id": "run-001"
}

--- Sample Log Record (Failure Run with Traceback) ---
Level:     ERROR
Error:     ProcessingError : Pipeline processing failed in NormalizeDataStep: cannot convert value 'INVALID_NUMERIC' for field 'complexity' to float.
Traceback snippet:
       return [self._normalize_record(record) for record in data]
     File "C:\Users\bc\task-management-api\src\task_analytics\pipeline.py", line 143, in _normalize_record
       raise ProcessingError(
   task_analytics.exceptions.ProcessingError: Pipeline processing failed in NormalizeDataStep: cannot convert value 'INVALID_NUMERIC' for field 'complexity' to float.


## Conclusion & Key Takeaways

- **Structured JSON**: Enables automated indexing and filtering in production monitoring platforms.
- **`logger.exception()`**: Automatically captures stack tracebacks for offline failure diagnosis.